In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

import matplotlib.pyplot as plt

import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
amazon="..\\data\\raw\\Reviews.csv"
tweet="..\\data\\raw\\training.1600000.processed.noemoticon.csv"

### Preprocessing of amazon dataset

In [3]:
#loading the data
df_amazon = pd.read_csv(amazon)

In [4]:
df_amazon.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [5]:
df_amazon = df_amazon[["Text", "Score"]]

# Binary sentiment
df_amazon["label"] = df_amazon["Score"].apply(
    lambda x: 1 if x >= 4 else 0
)

df_amazon = df_amazon[["Text", "label"]]

df_amazon = df_amazon.dropna()

df_amazon = df_amazon.sample(10000, random_state=42)

df_amazon.head()

,Text,label
165256,Having tried a couple of other brands of glute...,1
231465,My cat loves these treats. If ever I can't fin...,1
427827,A little less than I expected. It tends to ha...,0
433954,"First there was Frosted Mini-Wheats, in origin...",0
70260,and I want to congratulate the graphic artist ...,1


### preprocessing of tweets dataset

In [6]:
twitter_df = pd.read_csv(
    tweet,
    encoding="latin-1",
    header=None
)

In [7]:
twitter_df.head()

,0,1,2,3,4,5
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [8]:
twitter_df = twitter_df[[0, 5]]

twitter_df.columns = ["label", "Text"]

twitter_df["label"] = twitter_df["label"].map({
    0: 0,
    4: 1
})

twitter_df = twitter_df.dropna()

twitter_df = twitter_df.sample(10000, random_state=42)

twitter_df.head()

,label,Text
541200,0,@chrishasboobs AHHH I HOPE YOUR OK!!!
750,0,"@misstoriblack cool , i have no tweet apps fo..."
766711,0,@TiannaChaos i know just family drama. its la...
285055,0,School email won't open and I have geography ...
705995,0,upper airways problem


### Comparing processed and raw text

##### 1)Amazon

In [9]:
from src.preprocessing.textpreprocessing import (
    TextPreprocessor,
    PreprocessingConfig
)

processor = TextPreprocessor(
    PreprocessingConfig.for_reviews()
)

# preprocess text column
df = processor.process_dataframe(
    df=df_amazon,
    text_col="Text",
    out_col="clean_text"
)

print(df[["Text", "clean_text"]].head())

[food_reviews] preprocessing: 100%|██████████| 10000/10000 [00:10<00:00, 926.86it/s]

                                                     Text  \
165256  Having tried a couple of other brands of glute...   
231465  My cat loves these treats. If ever I can't fin...   
427827  A little less than I expected.  It tends to ha...   
433954  First there was Frosted Mini-Wheats, in origin...   
70260   and I want to congratulate the graphic artist ...   

                                               clean_text  
165256  tried couple brand glutenfree sandwich cooky b...  
231465  cat love treat ever not find house pop top bol...  
427827  little less expected tends muddy taste not exp...  
433954  first frosted miniwheats original size frosted...  
70260   want congratulate graphic artist putting entir...  


##### 2)tweets

In [10]:
from src.preprocessing.textpreprocessing import (
    TextPreprocessor,
    PreprocessingConfig
)

processor = TextPreprocessor(
    PreprocessingConfig.for_tweets()
)

# preprocess text column
df = processor.process_dataframe(
    df=twitter_df,
    text_col="Text",
    out_col="clean_text"
)

print(df[["Text", "clean_text"]].head())

[twitter] preprocessing: 100%|██████████| 10000/10000 [00:01<00:00, 6864.72it/s]
52 rows produced empty strings after preprocessing.


                                                     Text  \
541200             @chrishasboobs AHHH I HOPE YOUR OK!!!    
750     @misstoriblack cool , i have no tweet apps  fo...   
766711  @TiannaChaos i know  just family drama. its la...   
285055  School email won't open  and I have geography ...   
705995                             upper airways problem    

                                               clean_text  
541200                                          ahhh hope  
750                                  cool tweet apps razr  
766711  know family drama lamehey next time hang kim g...  
285055  school email not open geography stuff revise s...  
705995                               upper airway problem  


### Comparing different vectorization methods


#### 1) BOW

In [11]:
from src.vectorization.bow import BoWVectorizer
bow_vectorizer = BoWVectorizer(text_processor=processor)
X_amazon_bow = bow_vectorizer.fit_transform(df_amazon["clean_text"])
X_twitter_bow = bow_vectorizer.fit_transform(twitter_df["clean_text"])

In [12]:
print(X_amazon_bow.shape)
print(X_twitter_bow.shape)

(10000, 5000)
(10000, 5000)


#### 2)TF-IDF

In [13]:
from src.vectorization.tfidf import TFIDFVectorizer

tfidf_vectorizer = TFIDFVectorizer(text_processor=processor)
X_amazon_tfidf = tfidf_vectorizer.fit_transform(df_amazon["clean_text"])
X_twitter_tfidf = tfidf_vectorizer.fit_transform(twitter_df["clean_text"])
print(X_amazon_tfidf.shape)
print(X_twitter_tfidf.shape)

(10000, 5000)
(10000, 5000)


#### 3)bm-25

In [14]:
from src.vectorization.bm25 import BM25Vectorizer
bm25_vectorizer = BM25Vectorizer(text_processor=processor)
X_amazon_bm25 = bm25_vectorizer.fit(df_amazon["clean_text"])
X_twitter_bm25 = bm25_vectorizer.fit(twitter_df["clean_text"])

In [15]:
X_amazon_bm25.search("This product is great!", top_k=5)

(array([9127, 2358, 7898,    9,  123]),
 array([9.62166499, 6.64468248, 6.26701458, 6.25967173, 6.25967173]))

#### Embeddings

##### 1)Word2vec

In [16]:
from src.vectorization.embeddings import Embeddings
embeddings_vectorizer = Embeddings(embedding_type="word2vec",embedding_path=r"..\word2vec\GoogleNews-vectors-negative300.bin",vector_size=300,window=5,min_count=2,workers=4)
embeddings_vectorizer.load_embeddings()



In [17]:
vector = embeddings_vectorizer.most_similar("king")
print(vector)

[('kings', 0.7138045430183411), ('queen', 0.6510956883430481), ('monarch', 0.6413194537162781), ('crown_prince', 0.6204220056533813), ('prince', 0.6159993410110474)]


In [18]:
embeddings_vectorizer.train_embeddings(df_amazon["clean_text"])

In [19]:
print(
    embeddings_vectorizer.model.wv["good"]
)

[-1.60783514e-01  5.48837006e-01  2.82510202e-02  9.19658095e-02
 -1.64661482e-01 -6.25470877e-01  2.26332784e-01  1.16111779e+00
 -1.28765285e-01 -3.91811430e-01  2.00569451e-01 -5.19122422e-01
 -3.98725152e-01  3.96451317e-02 -1.84765771e-01 -3.94424409e-01
  4.66320276e-01  1.79410368e-01 -3.96327078e-02 -1.59432337e-01
  3.64675783e-02  7.17135370e-02  5.26190877e-01  3.51904303e-01
  3.58559638e-01 -8.19797441e-02 -4.04238939e-01 -3.52996551e-02
 -3.37035328e-01 -3.42885256e-01  3.60638678e-01 -1.83587641e-01
  3.23532701e-01  3.87037963e-01 -3.34840752e-02  2.50199884e-01
  3.47153991e-01 -6.01651847e-01  2.26844490e-01  2.79375523e-01
  2.12578494e-02  3.42702299e-01  1.09600760e-01 -7.23182380e-01
  4.00322407e-01  3.57334793e-01  4.58180271e-02  2.29016557e-01
  1.16526991e-01  3.67340118e-01  1.96327060e-01  1.75134823e-01
 -3.13654423e-01  5.36771752e-02 -5.68830185e-02  3.17653924e-01
  2.51523405e-01 -5.40973879e-02  3.40177745e-01  1.74026396e-02
 -2.50411272e-01  7.52054

In [20]:
print(
    embeddings_vectorizer.model.wv.most_similar("good", topn=5)
)

[('great', 0.9240363836288452), ('pretty', 0.8982052803039551), ('looking', 0.8893091678619385), ('restricted', 0.8839259743690491), ('overall', 0.867035984992981)]


##### 2)Glove vectorizer

In [21]:
from src.vectorization.embeddings import Embeddings
embeddings_vectorizer = Embeddings(embedding_type="glove",embedding_path=r"..\glove.6B\glove.6B.50d.txt",vector_size=300,window=5,min_count=2,workers=4)
embeddings_vectorizer.load_embeddings()
